<a href="https://colab.research.google.com/github/nguyenthuong5933-pixel/hethongnhung/blob/main/Train_YOLOv4_tt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Wed Oct  7 14:31:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content
!git clone https://github.com/AlexeyAB/darknet.git
%cd /content/darknet

/content
Cloning into 'darknet'...
remote: Enumerating objects: 15941, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 15941 (delta 13), reused 4 (delta 4), pack-reused 15914 (from 2)
Receiving objects: 100% (15941/15941), 14.52 MiB | 8.91 MiB/s, done.
Resolving deltas: 100% (10719/10719), done.
/content/darknet


In [3]:
!ls

3rdparty		data		       net_cam_v3.sh
build			docker-compose.yml     net_cam_v4.sh
CCM			Dockerfile.cpu	       package.xml
cfg			Dockerfile.gpu	       README.md
cmake			image_yolov3.sh        scripts
CMakeLists.txt		image_yolov4.sh        src
DarknetConfig.cmake.in	include		       vcpkg.json
darknet_images.py	json_mjpeg_streams.sh  video_yolov3.sh
darknet.py		LICENSE		       video_yolov4.sh
darknet_video.py	Makefile


In [4]:
%cd /content/darknet

!sed -i 's/^GPU=0/GPU=1/' Makefile
!sed -i 's/^CUDNN=0/CUDNN=1/' Makefile
!sed -i 's/^CUDNN_HALF=0/CUDNN_HALF=1/' Makefile
!sed -i 's/^OPENCV=0/OPENCV=1/' Makefile

!grep -E '^(GPU|CUDNN|CUDNN_HALF|OPENCV)=' Makefile

/content/darknet
GPU=1
CUDNN=1
CUDNN_HALF=1
OPENCV=1


In [5]:
!apt-get update -qq
!apt-get install -y libopencv-dev pkg-config

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-common at-spi2-core gsettings-desktop-schemas
  gstreamer1.0-plugins-base libatk-bridge2.0-0t64 libatk1.0-0t64
  libatspi2.0-0t64 libavcodec-dev libavformat-dev libavutil-dev libcdparanoia0
  libcharls2 libdc1394-dev libdouble-conversion3 libevdev2 libexif-dev
  libexif-doc libexif12 libgdcm-dev libgdcm3.0t64 libgl2ps1.4 libglew2.2
  libgphoto2-6t64 libgphoto2-dev libgphoto2-l10n libgphoto2-port12t64
  libgstreamer-plugins-base1.0-0 libgtk-3-0t64 libgtk-3-bin libgtk-3-common
  libgudev-1.0-0 libimath-3-1-29t64 libimath-dev libinput-bin libinput10
  libmd4c0 libmtdev1t64 libopencv-calib3d-dev libopencv-calib3d406t64
  libopencv-contrib-dev libo

In [6]:
!pkg-config --modversion opencv4

4.6.0


In [7]:
!cd /content/darknet && sed -i 's/-gencode arch=compute_50,code=\[sm_50,compute_50\] //' Makefile

In [8]:
!cd /content/darknet && make clean

rm -rf ./obj/image_opencv.o ./obj/http_stream.o ./obj/gemm.o ./obj/utils.o ./obj/dark_cuda.o ./obj/convolutional_layer.o ./obj/list.o ./obj/image.o ./obj/activations.o ./obj/im2col.o ./obj/col2im.o ./obj/blas.o ./obj/crop_layer.o ./obj/dropout_layer.o ./obj/maxpool_layer.o ./obj/softmax_layer.o ./obj/data.o ./obj/matrix.o ./obj/network.o ./obj/connected_layer.o ./obj/cost_layer.o ./obj/parser.o ./obj/option_list.o ./obj/darknet.o ./obj/detection_layer.o ./obj/captcha.o ./obj/route_layer.o ./obj/writing.o ./obj/box.o ./obj/nightmare.o ./obj/normalization_layer.o ./obj/avgpool_layer.o ./obj/coco.o ./obj/dice.o ./obj/yolo.o ./obj/detector.o ./obj/layer.o ./obj/compare.o ./obj/classifier.o ./obj/local_layer.o ./obj/swag.o ./obj/shortcut_layer.o ./obj/representation_layer.o ./obj/activation_layer.o ./obj/rnn_layer.o ./obj/gru_layer.o ./obj/rnn.o ./obj/rnn_vid.o ./obj/crnn_layer.o ./obj/demo.o ./obj/tag.o ./obj/cifar.o ./obj/go.o ./obj/batchnorm_layer.o ./obj/art.o ./obj/region_layer.o ./obj

In [9]:
!cd /content/darknet && make -j2

mkdir -p ./obj/
mkdir -p backup
mkdir -p results
chmod +x *.sh
g++ -std=c++11 -std=c++11 -Iinclude/ -I3rdparty/stb/include -DOPENCV `pkg-config --cflags opencv4 2> /dev/null || pkg-config --cflags opencv` -DGPU -I/usr/local/cuda/include/ -DCUDNN -DCUDNN_HALF -Wall -Wfatal-errors -Wno-unused-result -Wno-unknown-pragmas -fPIC -rdynamic -Ofast -DOPENCV -DGPU -DCUDNN -I/usr/local/cudnn/include -DCUDNN_HALF -c ./src/image_opencv.cpp -o obj/image_opencv.o
g++ -std=c++11 -std=c++11 -Iinclude/ -I3rdparty/stb/include -DOPENCV `pkg-config --cflags opencv4 2> /dev/null || pkg-config --cflags opencv` -DGPU -I/usr/local/cuda/include/ -DCUDNN -DCUDNN_HALF -Wall -Wfatal-errors -Wno-unused-result -Wno-unknown-pragmas -fPIC -rdynamic -Ofast -DOPENCV -DGPU -DCUDNN -I/usr/local/cudnn/include -DCUDNN_HALF -c ./src/http_stream.cpp -o obj/http_stream.o
./src/http_stream.cpp: In member function ‘bool JSON_sender::write(const char*)’:
./src/http_stream.cpp:253:21: warning: unused variable ‘n’ []8;;https://gc

In [10]:
!cd /content/darknet && sed -i 's/-gencode arch=compute_52,code=\[sm_52,compute_52\] //' Makefile
!cd /content/darknet && make clean
!cd /content/darknet && make -j2

rm -rf ./obj/image_opencv.o ./obj/http_stream.o ./obj/gemm.o ./obj/utils.o ./obj/dark_cuda.o ./obj/convolutional_layer.o ./obj/list.o ./obj/image.o ./obj/activations.o ./obj/im2col.o ./obj/col2im.o ./obj/blas.o ./obj/crop_layer.o ./obj/dropout_layer.o ./obj/maxpool_layer.o ./obj/softmax_layer.o ./obj/data.o ./obj/matrix.o ./obj/network.o ./obj/connected_layer.o ./obj/cost_layer.o ./obj/parser.o ./obj/option_list.o ./obj/darknet.o ./obj/detection_layer.o ./obj/captcha.o ./obj/route_layer.o ./obj/writing.o ./obj/box.o ./obj/nightmare.o ./obj/normalization_layer.o ./obj/avgpool_layer.o ./obj/coco.o ./obj/dice.o ./obj/yolo.o ./obj/detector.o ./obj/layer.o ./obj/compare.o ./obj/classifier.o ./obj/local_layer.o ./obj/swag.o ./obj/shortcut_layer.o ./obj/representation_layer.o ./obj/activation_layer.o ./obj/rnn_layer.o ./obj/gru_layer.o ./obj/rnn.o ./obj/rnn_vid.o ./obj/crnn_layer.o ./obj/demo.o ./obj/tag.o ./obj/cifar.o ./obj/go.o ./obj/batchnorm_layer.o ./obj/art.o ./obj/region_layer.o ./obj

In [11]:
!cd /content/darknet && sed -i 's/-gencode arch=compute_61,code=\[sm_61,compute_61\] //' Makefile
!cd /content/darknet && make clean
!cd /content/darknet && make -j2

rm -rf ./obj/image_opencv.o ./obj/http_stream.o ./obj/gemm.o ./obj/utils.o ./obj/dark_cuda.o ./obj/convolutional_layer.o ./obj/list.o ./obj/image.o ./obj/activations.o ./obj/im2col.o ./obj/col2im.o ./obj/blas.o ./obj/crop_layer.o ./obj/dropout_layer.o ./obj/maxpool_layer.o ./obj/softmax_layer.o ./obj/data.o ./obj/matrix.o ./obj/network.o ./obj/connected_layer.o ./obj/cost_layer.o ./obj/parser.o ./obj/option_list.o ./obj/darknet.o ./obj/detection_layer.o ./obj/captcha.o ./obj/route_layer.o ./obj/writing.o ./obj/box.o ./obj/nightmare.o ./obj/normalization_layer.o ./obj/avgpool_layer.o ./obj/coco.o ./obj/dice.o ./obj/yolo.o ./obj/detector.o ./obj/layer.o ./obj/compare.o ./obj/classifier.o ./obj/local_layer.o ./obj/swag.o ./obj/shortcut_layer.o ./obj/representation_layer.o ./obj/activation_layer.o ./obj/rnn_layer.o ./obj/gru_layer.o ./obj/rnn.o ./obj/rnn_vid.o ./obj/crnn_layer.o ./obj/demo.o ./obj/tag.o ./obj/cifar.o ./obj/go.o ./obj/batchnorm_layer.o ./obj/art.o ./obj/region_layer.o ./obj

In [12]:
!cd /content/darknet && sed -i 's/-gencode arch=compute_61,code=\[sm_61,compute_61\] //' Makefile

In [13]:
!grep -o "compute_[0-9]*" /content/darknet/Makefile | sort -u

compute_30
compute_52
compute_53
compute_60
compute_61
compute_62
compute_70
compute_72
compute_75
compute_80
compute_86
compute_87
compute_89
compute_90


In [14]:
!nvidia-smi

Wed Oct  7 14:40:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
!cd /content/darknet && sed -i '/-gencode arch=compute_/d' Makefile
!cd /content/darknet && sed -i '/NVCC=/a NVCCFLAGS += -gencode arch=compute_75,code=sm_75' Makefile
!grep -n "compute_\|sm_" /content/darknet/Makefile

79:NVCCFLAGS += -gencode arch=compute_75,code=sm_75


In [16]:
!cd /content/darknet && make clean
!cd /content/darknet && make -j2

rm -rf ./obj/image_opencv.o ./obj/http_stream.o ./obj/gemm.o ./obj/utils.o ./obj/dark_cuda.o ./obj/convolutional_layer.o ./obj/list.o ./obj/image.o ./obj/activations.o ./obj/im2col.o ./obj/col2im.o ./obj/blas.o ./obj/crop_layer.o ./obj/dropout_layer.o ./obj/maxpool_layer.o ./obj/softmax_layer.o ./obj/data.o ./obj/matrix.o ./obj/network.o ./obj/connected_layer.o ./obj/cost_layer.o ./obj/parser.o ./obj/option_list.o ./obj/darknet.o ./obj/detection_layer.o ./obj/captcha.o ./obj/route_layer.o ./obj/writing.o ./obj/box.o ./obj/nightmare.o ./obj/normalization_layer.o ./obj/avgpool_layer.o ./obj/coco.o ./obj/dice.o ./obj/yolo.o ./obj/detector.o ./obj/layer.o ./obj/compare.o ./obj/classifier.o ./obj/local_layer.o ./obj/swag.o ./obj/shortcut_layer.o ./obj/representation_layer.o ./obj/activation_layer.o ./obj/rnn_layer.o ./obj/gru_layer.o ./obj/rnn.o ./obj/rnn_vid.o ./obj/crnn_layer.o ./obj/demo.o ./obj/tag.o ./obj/cifar.o ./obj/go.o ./obj/batchnorm_layer.o ./obj/art.o ./obj/region_layer.o ./obj

In [17]:
!cd /content/darknet && ./darknet

usage: ./darknet <function>


In [18]:
from google.colab import files

uploaded = files.upload()

Saving data_yolo(1).zip to data_yolo(1).zip


In [20]:
!unzip -q "data_yolo(1).zip" -d /content/

In [21]:
!ls -lah "/content/data_yolo(1)"


total 23M
drwxrwxrwx 4 root root 4.0K Oct  7 14:26 .
drwxr-xr-x 1 root root 4.0K Oct  7 14:47 ..
-rw-rw-rw- 1 root root   40 Oct  4 16:28 classes.txt
drwxrwxrwx 4 root root 4.0K Oct  7 13:57 images
drwxrwxrwx 4 root root 4.0K Oct  7 13:57 labels
-rw-rw-rw- 1 root root  123 Oct  7 14:25 obj.data
-rw-rw-rw- 1 root root   40 Sep 29 16:23 obj.names
-rw-rw-rw- 1 root root 174K Oct  7 14:19 train.txt
-rw-rw-rw- 1 root root  42K Oct  7 14:19 val.txt
-rw-rw-rw- 1 root root 3.5K May 12  2023 yolov4-tiny.cfg
-rw-rw-rw- 1 root root  23M Jun 20  2023 yolov4-tiny_final_1806.weights


In [22]:
!cat "/content/data_yolo(1)/obj.data"

classes = 6
train = data_yolo(1)/train.txt
valid = data_yolo(1)/val.txt
names = data_yolo(1)/obj.names
backup = backup/

In [23]:
%%writefile "/content/data_yolo(1)/obj.data"
classes = 6
train = /content/data_yolo(1)/train.txt
valid = /content/data_yolo(1)/val.txt
names = /content/data_yolo(1)/obj.names
backup = /content/backup/

Overwriting /content/data_yolo(1)/obj.data


In [24]:
!cat "/content/data_yolo(1)/obj.data"

classes = 6
train = /content/data_yolo(1)/train.txt
valid = /content/data_yolo(1)/val.txt
names = /content/data_yolo(1)/obj.names
backup = /content/backup/


In [25]:
%cd /content/darknet
!./darknet

/content/darknet
usage: ./darknet <function>


In [26]:
!grep -E "classes=|filters=|max_batches|steps=" "/content/data_yolo(1)/yolov4-tiny.cfg"

max_batches = 6000
steps=4800,5400
filters=32
filters=64
filters=64
filters=32
filters=32
filters=64
filters=128
filters=64
filters=64
filters=128
filters=256
filters=128
filters=128
filters=256
filters=512
filters=256
filters=512
filters=33
classes=6
filters=128
filters=256
filters=33
classes=6


In [27]:
!sed -i 's/max_batches = 6000/max_batches = 8000/' "/content/data_yolo(1)/yolov4-tiny.cfg"
!sed -i 's/steps=4800,5400/steps=6400,7200/' "/content/data_yolo(1)/yolov4-tiny.cfg"

In [28]:
!grep -E "max_batches|steps=|filters=33|classes=6" "/content/data_yolo(1)/yolov4-tiny.cfg"

max_batches = 8000
steps=6400,7200
filters=33
classes=6
filters=33
classes=6


In [29]:
!mkdir -p /content/backup

In [30]:
!ls -lh "/content/data_yolo(1)/yolov4-tiny_final_1806.weights"

-rw-rw-rw- 1 root root 23M Jun 20  2023 '/content/data_yolo(1)/yolov4-tiny_final_1806.weights'


In [33]:
!sed -i 's/^batch=1/batch=64/' "/content/data_yolo(1)/yolov4-tiny.cfg"
!sed -i 's/^subdivisions=1/subdivisions=64/' "/content/data_yolo(1)/yolov4-tiny.cfg"
!grep -E "^batch=|^subdivisions=|max_batches|steps=" "/content/data_yolo(1)/yolov4-tiny.cfg"

batch=64
subdivisions=64
max_batches = 8000
steps=6400,7200


In [35]:
!sed -i 's#data_yolo(1)/#/content/data_yolo(1)/#g' "/content/data_yolo(1)/train.txt"
!sed -i 's#data_yolo(1)/#/content/data_yolo(1)/#g' "/content/data_yolo(1)/val.txt"
!head -5 "/content/data_yolo(1)/train.txt"

/content/data_yolo(1)/images/train/Close(1).jpg
/content/data_yolo(1)/images/train/Close(10).jpg
/content/data_yolo(1)/images/train/Close(100).jpg
/content/data_yolo(1)/images/train/Close(1002).jpg
/content/data_yolo(1)/images/train/Close(1003).jpg


In [36]:
import os

p = "/content/data_yolo(1)/images/train/Down(338).jpg"
print(os.path.exists(p))

True


In [38]:
import os

img = "/content/data_yolo(1)/images/train/Down(357).jpg"
label = "/content/data_yolo(1)/labels/train/Down(357).txt"

print("Ảnh :", os.path.exists(img))
print("Label:", os.path.exists(label))

Ảnh : True
Label: True


In [39]:
!ls -l "/content/data_yolo(1)/labels/train/Down(357).txt"
!ls -l "/content/data_yolo(1)/images/train/Down(357).jpg"

-rw-rw-rw- 1 root root 38 Oct  4 06:00 '/content/data_yolo(1)/labels/train/Down(357).txt'
-rw-rw-rw- 1 root root 5281 Oct  3 06:47 '/content/data_yolo(1)/images/train/Down(357).jpg'


In [40]:
!cp /content/data_yolo\(1\)/labels/train/*.txt /content/data_yolo\(1\)/images/train/
!cp /content/data_yolo\(1\)/labels/val/*.txt /content/data_yolo\(1\)/images/val/

In [41]:
!ls -l "/content/data_yolo(1)/images/train/Down(357).txt"

-rw-r--r-- 1 root root 38 Oct  7 15:08 '/content/data_yolo(1)/images/train/Down(357).txt'


In [42]:
!echo "Label gốc:"
!find "/content/data_yolo(1)/labels/train" -name "*.txt" | wc -l

!echo "Label copy:"
!find "/content/data_yolo(1)/images/train" -name "*.txt" | wc -l

Label gốc:
4256
Label copy:
4256


In [43]:
%cd /content/darknet

!./darknet detector train \
"/content/data_yolo(1)/obj.data" \
"/content/data_yolo(1)/yolov4-tiny.cfg" \
"/content/data_yolo(1)/yolov4-tiny_final_1806.weights" \
-dont_show

Streaming output truncated to the last 5000 lines.
 total_bbox = 126621, rewritten_bbox = 0.000000 % 
v3 (iou loss, Normalizer: (iou: 0.07, obj: 1.00, cls: 1.00) Region 30 Avg (IOU: 0.916455), count: 1, class_loss = 0.000550, iou_loss = 0.235120, total_loss = 0.235670 
v3 (iou loss, Normalizer: (iou: 0.07, obj: 1.00, cls: 1.00) Region 37 Avg (IOU: 0.000000), count: 1, class_loss = 0.000000, iou_loss = 0.000000, total_loss = 0.000000 
 total_bbox = 126622, rewritten_bbox = 0.000000 % 
v3 (iou loss, Normalizer: (iou: 0.07, obj: 1.00, cls: 1.00) Region 30 Avg (IOU: 0.885862), count: 1, class_loss = 0.216592, iou_loss = 0.270433, total_loss = 0.487025 
v3 (iou loss, Normalizer: (iou: 0.07, obj: 1.00, cls: 1.00) Region 37 Avg (IOU: 0.000000), count: 1, class_loss = 0.000001, iou_loss = 0.000000, total_loss = 0.000001 
 total_bbox = 126623, rewritten_bbox = 0.000000 % 
v3 (iou loss, Normalizer: (iou: 0.07, obj: 1.00, cls: 1.00) Region 30 Avg (IOU: 0.854189), count: 1, class_loss = 0.005301, 

In [44]:
!ls -lh /content/backup/yolov4-tiny_final.weights

-rw-r--r-- 1 root root 23M Oct  7 15:38 /content/backup/yolov4-tiny_final.weights


In [45]:
%cd /content/darknet

!./darknet detector map \
"/content/data_yolo(1)/obj.data" \
"/content/data_yolo(1)/yolov4-tiny.cfg" \
"/content/backup/yolov4-tiny_final.weights" \
-i 0 \
-thresh 0.25

/content/darknet
 CUDA-version: 13000 (13000), cuDNN: 9.14.0, CUDNN_HALF=1, GPU count: 1  
 CUDNN_HALF=1 
 OpenCV version: 4.6.0
 0 : compute_capability = 750, cudnn_half = 1, GPU: Tesla T4 
net.optimized_memory = 0 
mini_batch = 1, batch = 64, time_steps = 1, train = 0 
   layer   filters  size/strd(dil)      input                output
   0 Create CUDA-stream - 0 
 Create cudnn-handle 0 
conv     32       3 x 3/ 2    416 x 416 x   3 ->  208 x 208 x  32 0.075 BF
   1 conv     64       3 x 3/ 2    208 x 208 x  32 ->  104 x 104 x  64 0.399 BF
   2 conv     64       3 x 3/ 1    104 x 104 x  64 ->  104 x 104 x  64 0.797 BF
   3 route  2 		                       1/2 ->  104 x 104 x  32 
   4 conv     32       3 x 3/ 1    104 x 104 x  32 ->  104 x 104 x  32 0.199 BF
   5 conv     32       3 x 3/ 1    104 x 104 x  32 ->  104 x 104 x  32 0.199 BF
   6 route  5 4 	                           ->  104 x 104 x  64 
   7 conv     64       1 x 1/ 1    104 x 104 x  64 ->  104 x 104 x  64 0.089 BF
   

In [46]:
%cd /content/darknet

!./darknet detector map \
"/content/data_yolo(1)/obj.data" \
"/content/data_yolo(1)/yolov4-tiny.cfg" \
"/content/backup/yolov4-tiny_final.weights" \
-i 0 \
-thresh 0.25

/content/darknet
 CUDA-version: 13000 (13000), cuDNN: 9.14.0, CUDNN_HALF=1, GPU count: 1  
 CUDNN_HALF=1 
 OpenCV version: 4.6.0
 0 : compute_capability = 750, cudnn_half = 1, GPU: Tesla T4 
net.optimized_memory = 0 
mini_batch = 1, batch = 64, time_steps = 1, train = 0 
   layer   filters  size/strd(dil)      input                output
   0 Create CUDA-stream - 0 
 Create cudnn-handle 0 
conv     32       3 x 3/ 2    416 x 416 x   3 ->  208 x 208 x  32 0.075 BF
   1 conv     64       3 x 3/ 2    208 x 208 x  32 ->  104 x 104 x  64 0.399 BF
   2 conv     64       3 x 3/ 1    104 x 104 x  64 ->  104 x 104 x  64 0.797 BF
   3 route  2 		                       1/2 ->  104 x 104 x  32 
   4 conv     32       3 x 3/ 1    104 x 104 x  32 ->  104 x 104 x  32 0.199 BF
   5 conv     32       3 x 3/ 1    104 x 104 x  32 ->  104 x 104 x  32 0.199 BF
   6 route  5 4 	                           ->  104 x 104 x  64 
   7 conv     64       1 x 1/ 1    104 x 104 x  64 ->  104 x 104 x  64 0.089 BF
   

In [47]:
%cd /content/darknet

!ls -lh /content/backup/yolov4-tiny_final.weights

/content/darknet
-rw-r--r-- 1 root root 23M Oct  7 15:38 /content/backup/yolov4-tiny_final.weights


In [48]:
%cd /content/darknet

!./darknet detector test \
"/content/data_yolo(1)/obj.data" \
"/content/data_yolo(1)/yolov4-tiny.cfg" \
"/content/backup/yolov4-tiny_final.weights" \
"/content/data_yolo(1)/val.txt" \
-thresh 0.25 \
-ext_output \
-dont_show \
-out_filename "/content/predictions.json"

/content/darknet
 CUDA-version: 13000 (13000), cuDNN: 9.14.0, CUDNN_HALF=1, GPU count: 1  
 CUDNN_HALF=1 
 OpenCV version: 4.6.0
 0 : compute_capability = 750, cudnn_half = 1, GPU: Tesla T4 
net.optimized_memory = 0 
mini_batch = 1, batch = 64, time_steps = 1, train = 0 
   layer   filters  size/strd(dil)      input                output
   0 Create CUDA-stream - 0 
 Create cudnn-handle 0 
conv     32       3 x 3/ 2    416 x 416 x   3 ->  208 x 208 x  32 0.075 BF
   1 conv     64       3 x 3/ 2    208 x 208 x  32 ->  104 x 104 x  64 0.399 BF
   2 conv     64       3 x 3/ 1    104 x 104 x  64 ->  104 x 104 x  64 0.797 BF
   3 route  2 		                       1/2 ->  104 x 104 x  32 
   4 conv     32       3 x 3/ 1    104 x 104 x  32 ->  104 x 104 x  32 0.199 BF
   5 conv     32       3 x 3/ 1    104 x 104 x  32 ->  104 x 104 x  32 0.199 BF
   6 route  5 4 	                           ->  104 x 104 x  64 
   7 conv     64       1 x 1/ 1    104 x 104 x  64 ->  104 x 104 x  64 0.089 BF
   

In [50]:
%cd /content/darknet

!./darknet detector test -h

/content/darknet
 CUDA-version: 13000 (13000), cuDNN: 9.14.0, CUDNN_HALF=1, GPU count: 1  
 CUDNN_HALF=1 
 OpenCV version: 4.6.0
Couldn't open file: -h


In [51]:
import cv2

cfg = "/content/data_yolo(1)/yolov4-tiny.cfg"
weights = "/content/backup/yolov4-tiny_final.weights"

net = cv2.dnn.readNetFromDarknet(cfg, weights)

print("Đã load model thành công!")
print("Số layer:", len(net.getLayerNames()))

Đã load model thành công!
Số layer: 78
